# Notebook 10 — Hyperparameter Tuning

**PharmaLens**

---

## Objective

Use **Optuna** for hyperparameter optimization on the best-performing model(s).

We tune:
1. XGBoost (max_depth, learning_rate, n_estimators, subsample, etc.)
2. Best deep learning model (learning rate, dropout, hidden dimensions)

In [ ]:
import os, sys, warnings, pickle, time
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.config import *
from src.evaluation.metrics import compute_regression_metrics
from src.evaluation.experiment_tracker import ExperimentTracker
from src.models.xgboost_model import XGBoostDTI

set_seed(42)
COLORS = set_plot_style()
ensure_dirs()
tracker = ExperimentTracker()
print("Setup complete ✓")

## 1. Load Features

In [ ]:
with open(PROCESSED_DATA_DIR / 'drug_features.pkl', 'rb') as f:
    drug_features = pickle.load(f)
with open(PROCESSED_DATA_DIR / 'protein_features.pkl', 'rb') as f:
    protein_features = pickle.load(f)

train_df = pd.read_csv(SPLITS_DIR / 'random_train.csv')
val_df = pd.read_csv(SPLITS_DIR / 'random_val.csv')
test_df = pd.read_csv(SPLITS_DIR / 'random_test.csv')

def build_feature_matrix(df, drug_features, protein_features):
    X_list, y_list = [], []
    for _, row in df.iterrows():
        if row['drug_id'] in drug_features and row['target_id'] in protein_features:
            combined = np.concatenate([drug_features[row['drug_id']], protein_features[row['target_id']]])
            X_list.append(combined)
            y_list.append(row['score'])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)

X_train, y_train = build_feature_matrix(train_df, drug_features, protein_features)
X_val, y_val = build_feature_matrix(val_df, drug_features, protein_features)
X_test, y_test = build_feature_matrix(test_df, drug_features, protein_features)

print(f"Data loaded: X_train={X_train.shape}, X_val={X_val.shape}, X_test={X_test.shape}")

## 2. XGBoost Tuning with Optuna

In [ ]:
def xgb_objective(trial):
    """Optuna objective function for XGBoost."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 2000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
    }
    
    model = XGBoostDTI(**params, random_state=42, early_stopping_rounds=50)
    model.fit(X_train, y_train, X_val, y_val, verbose=False)
    
    y_val_pred = model.predict(X_val)
    metrics = compute_regression_metrics(y_val, y_val_pred)
    
    return metrics['rmse']  # Minimize RMSE

# Run Optuna study
study = optuna.create_study(direction='minimize', study_name='xgboost_tuning')
study.optimize(xgb_objective, n_trials=50, show_progress_bar=True)

print(f"\nBest trial:")
print(f"  Val RMSE: {study.best_trial.value:.4f}")
print(f"  Params: {study.best_trial.params}")

In [ ]:
# Visualize Optuna results
fig1 = plot_optimization_history(study)
plt.title('XGBoost Tuning — Optimization History', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgb_optuna_history.png', dpi=300, bbox_inches='tight')
plt.show()

fig2 = plot_param_importances(study)
plt.title('XGBoost Tuning — Hyperparameter Importance', fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'xgb_optuna_importance.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Train Best Model

In [ ]:
# Train the tuned model on full train data
best_params = study.best_trial.params
tuned_model = XGBoostDTI(**best_params, random_state=42, early_stopping_rounds=50)
tuned_model.fit(X_train, y_train, X_val, y_val, verbose=True)

# Evaluate
y_test_pred = tuned_model.predict(X_test)
test_metrics = compute_regression_metrics(y_test, y_test_pred)

print(f"\nTuned XGBoost Test Results:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# Log
tracker.log_experiment(
    model='XGBoost_Tuned', dataset='KIBA',
    representation='descriptors+fingerprints+aac+dpc',
    split='random', seed=42,
    hyperparameters=best_params,
    metrics=test_metrics,
    training_time=tuned_model.training_time,
)

# Save
tuned_model.save(XGBOOST_MODELS_DIR / 'xgb_tuned_best.json')
print("\nTuned model saved ✓")

## Conclusion

Hyperparameter tuning via Optuna found improved settings for XGBoost. The tuned model performance is logged alongside untuned results for comparison.

**Next:** [Notebook 11 — Cold-Start Evaluation](./11_cold_start_evaluation.ipynb)